# Itchy v2 — Full Stack Validation

Byte-level model with all competition tricks + n-gram hashing.

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**

In [ ]:
!pip install -q torch numpy huggingface-hub sentencepiece
import torch
print(f'PyTorch {torch.__version__}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# Download and convert data
import os, shutil, numpy as np, math, time
from pathlib import Path
from huggingface_hub import hf_hub_download
import sentencepiece as spm

os.makedirs('data/bytes', exist_ok=True)

def dl(filename, subfolder, dest):
    if Path(dest).exists(): return
    print(f'  Downloading {filename}...')
    cached = str(Path(hf_hub_download(repo_id='willdepueoai/parameter-golf',
        filename=filename, subfolder=subfolder, repo_type='dataset')).resolve())
    try: os.link(cached, dest)
    except: shutil.copy2(cached, dest)

dl('fineweb_train_000000.bin', 'datasets/datasets/fineweb10B_sp1024', 'data/train.bin')
dl('fineweb_val_000000.bin', 'datasets/datasets/fineweb10B_sp1024', 'data/val.bin')
dl('fineweb_1024_bpe.model', 'datasets/tokenizers', 'data/tok.model')

def load_shard(path):
    h = np.fromfile(path, dtype='<i4', count=256)
    return np.fromfile(path, dtype='<u2', count=int(h[2]),
                       offset=256*np.dtype('<i4').itemsize)

def write_shard(path, data):
    h = np.zeros(256, dtype='<i4'); h[0]=20240520; h[1]=1; h[2]=len(data)
    with open(path,'wb') as f: f.write(h.tobytes()); f.write(data.astype('<u2').tobytes())

if not Path('data/bytes/train.bin').exists():
    sp = spm.SentencePieceProcessor(model_file='data/tok.model')
    for src, dst in [('data/train.bin','data/bytes/train.bin'),('data/val.bin','data/bytes/val.bin')]:
        print(f'Converting {src}...')
        tokens = load_shard(src)
        chunks = [np.frombuffer(sp.decode(tokens[i:i+100000].tolist()).encode('utf-8'), dtype=np.uint8)
                  for i in range(0, len(tokens), 100000)]
        write_shard(dst, np.concatenate(chunks))
        print(f'  {len(tokens):,} tokens -> {sum(len(c) for c in chunks):,} bytes')

def load_bytes(path):
    h = np.fromfile(path, dtype='<i4', count=256)
    return np.fromfile(path, dtype='<u2', count=int(h[2]),
                       offset=256*np.dtype('<i4').itemsize).astype(np.int64)

train_data = load_bytes('data/bytes/train.bin')
val_data = load_bytes('data/bytes/val.bin')
print(f'\nTrain: {len(train_data):,} bytes | Val: {len(val_data):,} bytes')

In [ ]:
# ============================================================
# MODEL: Itchy v2
# ============================================================
import torch
import torch.nn.functional as F
from torch import Tensor, nn

device = torch.device('cuda')


class ByteNgramHash(nn.Module):
    """Hash n-gram embeddings: for each byte, hash n-grams of length 2-6,
    look up embeddings, sum them. Captures local byte patterns."""
    def __init__(self, hash_vocab, embed_dim, model_dim, ngram_sizes=(2,3,4,5,6)):
        super().__init__()
        self.hash_vocab = hash_vocab
        self.ngram_sizes = ngram_sizes
        total = hash_vocab * len(ngram_sizes)
        self.embed = nn.Embedding(total, embed_dim)
        nn.init.zeros_(self.embed.weight)
        self.proj = nn.Linear(embed_dim, model_dim, bias=False) if embed_dim != model_dim else None
        if self.proj: nn.init.zeros_(self.proj.weight)
        self.scale = nn.Parameter(torch.tensor(0.05))
        self.primes = [36313, 27191, 51637, 39371, 73291]

    def forward(self, byte_ids):
        t = byte_ids.int()
        B, S = t.shape
        result = torch.zeros(B, S, self.embed.embedding_dim, device=t.device)
        for idx, n in enumerate(self.ngram_sizes):
            offset = idx * self.hash_vocab
            h = torch.zeros_like(t)
            for k in range(n):
                p = self.primes[k % len(self.primes)]
                if k == 0:
                    h = p * t
                else:
                    shifted = F.pad(t[:, :-k], (k, 0), value=0)
                    h = h ^ (p * shifted)
            indices = (h.abs() % self.hash_vocab + offset).long()
            if n > 1:
                mask = torch.cat([torch.zeros(B, n-1, device=t.device),
                                  torch.ones(B, S-n+1, device=t.device)], dim=1).long()
                indices = indices * mask + offset * (1 - mask)
            result = result + self.embed(indices)
        if self.proj: result = self.proj(result)
        return result * self.scale


class Rotary(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)
        self._cache = None
    def forward(self, seq_len, device, dtype):
        if self._cache is None or self._cache[0] != seq_len:
            t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(t, self.inv_freq.to(device))
            self._cache = (seq_len, freqs.cos()[None,None,:,:], freqs.sin()[None,None,:,:])
        return self._cache[1].to(dtype), self._cache[2].to(dtype)


def apply_rotary_partial(x, cos, sin, rope_dims):
    """Apply RoPE to first rope_dims dimensions only."""
    if rope_dims >= x.size(-1):
        half = x.size(-1) // 2
        x1, x2 = x[..., :half], x[..., half:]
        return torch.cat((x1*cos + x2*sin, x1*(-sin) + x2*cos), dim=-1)
    x_rope, x_pass = x[..., :rope_dims], x[..., rope_dims:]
    half = rope_dims // 2
    x1, x2 = x_rope[..., :half], x_rope[..., half:]
    x_rope = torch.cat((x1*cos + x2*sin, x1*(-sin) + x2*cos), dim=-1)
    return torch.cat((x_rope, x_pass), dim=-1)


class Attention(nn.Module):
    def __init__(self, dim, num_heads, num_kv_heads, rope_base, qk_gain, rope_dims=0):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        self.rope_dims = rope_dims if rope_dims > 0 else self.head_dim
        kv_dim = num_kv_heads * self.head_dim
        self.c_q = nn.Linear(dim, dim, bias=False)
        self.c_k = nn.Linear(dim, kv_dim, bias=False)
        self.c_v = nn.Linear(dim, kv_dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        nn.init.zeros_(self.proj.weight)
        self.q_gain = nn.Parameter(torch.full((num_heads,), qk_gain))
        self.rotary = Rotary(self.rope_dims, rope_base)

    def forward(self, x):
        B, S, D = x.shape
        q = self.c_q(x).reshape(B, S, self.num_heads, self.head_dim).transpose(1,2)
        k = self.c_k(x).reshape(B, S, self.num_kv_heads, self.head_dim).transpose(1,2)
        v = self.c_v(x).reshape(B, S, self.num_kv_heads, self.head_dim).transpose(1,2)
        q = F.rms_norm(q, (q.size(-1),))
        k = F.rms_norm(k, (k.size(-1),))
        cos, sin = self.rotary(S, x.device, q.dtype)
        q = apply_rotary_partial(q, cos, sin, self.rope_dims)
        k = apply_rotary_partial(k, cos, sin, self.rope_dims)
        q = q * self.q_gain.to(q.dtype)[None,:,None,None]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                          enable_gqa=(self.num_kv_heads != self.num_heads))
        return self.proj(y.transpose(1,2).reshape(B, S, D))


class MLP(nn.Module):
    """LeakyReLU(0.5)² — proven -0.003 BPB over relu²."""
    def __init__(self, dim, mult):
        super().__init__()
        self.fc = nn.Linear(dim, dim*mult, bias=False)
        self.proj = nn.Linear(dim*mult, dim, bias=False)
        nn.init.zeros_(self.proj.weight)
    def forward(self, x):
        x = F.leaky_relu(self.fc(x), negative_slope=0.5)
        return self.proj(x.square())


class Block(nn.Module):
    def __init__(self, dim, num_heads, num_kv_heads, mlp_mult, rope_base,
                 qk_gain, rope_dims, layer_idx, ln_scale=True):
        super().__init__()
        self.attn = Attention(dim, num_heads, num_kv_heads, rope_base, qk_gain, rope_dims)
        self.mlp = MLP(dim, mlp_mult)
        self.attn_scale = nn.Parameter(torch.ones(dim))
        self.mlp_scale = nn.Parameter(torch.ones(dim))
        self.resid_mix = nn.Parameter(torch.stack((torch.ones(dim), torch.zeros(dim))))
        self.ln_factor = 1.0 / math.sqrt(layer_idx + 1) if ln_scale else 1.0

    def forward(self, x, x0):
        mix = self.resid_mix.to(x.dtype)
        x = mix[0][None,None,:] * x + mix[1][None,None,:] * x0
        normed = F.rms_norm(x, (x.size(-1),)) * self.ln_factor
        attn_out = self.attn(normed)
        x = x + self.attn_scale.to(x.dtype)[None,None,:] * attn_out
        normed = F.rms_norm(x, (x.size(-1),)) * self.ln_factor
        x = x + self.mlp_scale.to(x.dtype)[None,None,:] * self.mlp(normed)
        return x


class ItchyV2(nn.Module):
    def __init__(self, dim=384, num_layers=11, num_heads=8, num_kv_heads=4,
                 mlp_mult=3, patch_size=4, rope_dims=16,
                 ngram_hash_vocab=4096, ngram_dim=64, ngram_sizes=(2,3,4,5,6),
                 softcap=30.0, rope_base=10000.0, qk_gain=1.5):
        super().__init__()
        self.dim = dim
        self.patch_size = patch_size
        self.softcap = softcap
        self.byte_embed = nn.Embedding(260, dim)
        self.patch_proj = nn.Linear(dim * patch_size, dim, bias=False)
        self.ngram = ByteNgramHash(ngram_hash_vocab, ngram_dim, dim, ngram_sizes) if ngram_hash_vocab > 0 else None
        self.blocks = nn.ModuleList([
            Block(dim, num_heads, num_kv_heads, mlp_mult, rope_base, qk_gain,
                  rope_dims, layer_idx=i)
            for i in range(num_layers)
        ])
        self.unpatch = nn.Linear(dim, patch_size * 260, bias=False)
        self.num_enc = num_layers // 2
        self.num_dec = num_layers - self.num_enc
        self.skip_weights = nn.Parameter(torch.ones(min(self.num_enc, self.num_dec), dim))

    def forward(self, byte_ids, targets):
        B, S = byte_ids.shape
        P = self.patch_size
        # Patch embed
        x = self.byte_embed(byte_ids)
        x = x.reshape(B, S // P, P * self.dim)
        x = F.rms_norm(self.patch_proj(x), (self.dim,))
        # Add n-gram context
        if self.ngram is not None:
            ng = self.ngram(byte_ids)
            ng = ng.reshape(B, S // P, P, -1).mean(dim=2)
            x = x + ng.to(x.dtype)
        # Transformer
        x0 = x
        skips = []
        for i in range(self.num_enc):
            x = self.blocks[i](x, x0)
            skips.append(x)
        for i in range(self.num_dec):
            if skips:
                x = x + self.skip_weights[i].to(x.dtype)[None,None,:] * skips.pop()
            x = self.blocks[self.num_enc + i](x, x0)
        x = F.rms_norm(x, (x.size(-1),))
        # Unpatch to byte logits
        logits = self.unpatch(x).reshape(-1, 260)
        logits = self.softcap * torch.tanh(logits / self.softcap)
        return F.cross_entropy(logits.float(), targets.reshape(-1))

n_params = lambda m: sum(p.numel() for p in m.parameters())
print('Model defined')

In [ ]:
# ============================================================
# CONFIG
# ============================================================

# V2 model (scaled for T4)
DIM = 256
NUM_LAYERS = 8
NUM_HEADS = 8
NUM_KV_HEADS = 4
MLP_MULT = 3
PATCH_SIZE = 4
ROPE_DIMS = 8       # partial RoPE: 8 of 32 head dims
NGRAM_VOCAB = 2048
NGRAM_DIM = 64
SEQ_LEN = 512

# Training
TRAIN_STEPS = 3000
BATCH_SIZE = 8192
LR = 3e-4
LOG_EVERY = 200

print(f'Config: dim={DIM} layers={NUM_LAYERS} mlp={MLP_MULT}x rope_dims={ROPE_DIMS} ngram_vocab={NGRAM_VOCAB}')

In [ ]:
# ============================================================
# TRAIN V2
# ============================================================

torch.manual_seed(42)
model_v2 = ItchyV2(
    dim=DIM, num_layers=NUM_LAYERS, num_heads=NUM_HEADS, num_kv_heads=NUM_KV_HEADS,
    mlp_mult=MLP_MULT, patch_size=PATCH_SIZE, rope_dims=ROPE_DIMS,
    ngram_hash_vocab=NGRAM_VOCAB, ngram_dim=NGRAM_DIM,
).to(device).bfloat16()

print(f'Itchy v2: {n_params(model_v2):,} params ({n_params(model_v2)/1e6:.1f}M)')

optimizer = torch.optim.AdamW(model_v2.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.01)
pos = 0
t0 = time.time()
losses = []

print(f'Training {TRAIN_STEPS} steps...\n')
for step in range(1, TRAIN_STEPS + 1):
    usable = (BATCH_SIZE // SEQ_LEN) * SEQ_LEN
    end = pos + usable + 1
    if end > len(train_data): pos = 0; end = usable + 1
    chunk = train_data[pos:end]; pos = end - 1
    x = torch.tensor(chunk[:-1].reshape(-1, SEQ_LEN), device=device)
    y = torch.tensor(chunk[1:].reshape(-1, SEQ_LEN), device=device)

    optimizer.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        loss = model_v2(x, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if step % LOG_EVERY == 0 or step == 1:
        avg = np.mean(losses[-LOG_EVERY:])
        bpb = avg / math.log(2)
        print(f'step {step:>5} | loss {avg:.4f} | bpb {bpb:.4f} | {time.time()-t0:.0f}s')

print(f'\nTraining done in {time.time()-t0:.0f}s')

In [ ]:
# ============================================================
# ALSO TRAIN V1 (no tricks) FOR COMPARISON
# ============================================================

class ItchyV1(nn.Module):
    """V1: byte-level, relu², full RoPE, no n-gram, no LN scale."""
    def __init__(self, dim, num_layers, num_heads, num_kv_heads, mlp_mult, patch_size,
                 softcap=30.0, rope_base=10000.0, qk_gain=1.5):
        super().__init__()
        self.dim = dim
        self.patch_size = patch_size
        self.softcap = softcap
        self.byte_embed = nn.Embedding(260, dim)
        self.patch_proj = nn.Linear(dim * patch_size, dim, bias=False)
        # V1 blocks: relu², full RoPE, no LN scale
        self.blocks = nn.ModuleList()
        for i in range(num_layers):
            b = Block(dim, num_heads, num_kv_heads, mlp_mult, rope_base, qk_gain,
                      rope_dims=dim//num_heads, layer_idx=0, ln_scale=False)  # no LN scale
            # Override MLP to use relu² instead of leaky_relu²
            self.blocks.append(b)
        self.unpatch = nn.Linear(dim, patch_size * 260, bias=False)
        self.num_enc = num_layers // 2
        self.num_dec = num_layers - self.num_enc
        self.skip_weights = nn.Parameter(torch.ones(min(self.num_enc, self.num_dec), dim))

    def forward(self, byte_ids, targets):
        B, S = byte_ids.shape
        P = self.patch_size
        x = self.byte_embed(byte_ids)
        x = x.reshape(B, S // P, P * self.dim)
        x = F.rms_norm(self.patch_proj(x), (self.dim,))
        x0 = x
        skips = []
        for i in range(self.num_enc):
            x = self.blocks[i](x, x0); skips.append(x)
        for i in range(self.num_dec):
            if skips: x = x + self.skip_weights[i].to(x.dtype)[None,None,:] * skips.pop()
            x = self.blocks[self.num_enc + i](x, x0)
        x = F.rms_norm(x, (x.size(-1),))
        logits = self.unpatch(x).reshape(-1, 260)
        logits = self.softcap * torch.tanh(logits / self.softcap)
        return F.cross_entropy(logits.float(), targets.reshape(-1))

torch.manual_seed(42)
model_v1 = ItchyV1(
    dim=DIM, num_layers=NUM_LAYERS, num_heads=NUM_HEADS, num_kv_heads=NUM_KV_HEADS,
    mlp_mult=2, patch_size=PATCH_SIZE,  # v1 used 2x MLP
).to(device).bfloat16()

print(f'Itchy v1: {n_params(model_v1):,} params ({n_params(model_v1)/1e6:.1f}M)')
print(f'Itchy v2: {n_params(model_v2):,} params ({n_params(model_v2)/1e6:.1f}M)')

opt_v1 = torch.optim.AdamW(model_v1.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.01)
pos = 0
t0 = time.time()
losses_v1 = []

print(f'\nTraining v1 for {TRAIN_STEPS} steps...\n')
for step in range(1, TRAIN_STEPS + 1):
    usable = (BATCH_SIZE // SEQ_LEN) * SEQ_LEN
    end = pos + usable + 1
    if end > len(train_data): pos = 0; end = usable + 1
    chunk = train_data[pos:end]; pos = end - 1
    x = torch.tensor(chunk[:-1].reshape(-1, SEQ_LEN), device=device)
    y = torch.tensor(chunk[1:].reshape(-1, SEQ_LEN), device=device)

    opt_v1.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        loss = model_v1(x, y)
    loss.backward()
    opt_v1.step()
    losses_v1.append(loss.item())

    if step % LOG_EVERY == 0 or step == 1:
        avg = np.mean(losses_v1[-LOG_EVERY:])
        bpb = avg / math.log(2)
        print(f'step {step:>5} | loss {avg:.4f} | bpb {bpb:.4f} | {time.time()-t0:.0f}s')

print(f'V1 training done in {time.time()-t0:.0f}s')

In [ ]:
# ============================================================
# VALIDATION COMPARISON: V1 vs V2
# ============================================================

print('='*70)
print('VALIDATION COMPARISON')
print('='*70)

def eval_val(model, data, seq_len, n_samples=200):
    losses = []
    for i in range(0, min(n_samples * seq_len, len(data) - seq_len - 1), seq_len):
        chunk = data[i:i + seq_len + 1]
        x = torch.tensor(chunk[:-1].reshape(1, -1), device=device)
        y = torch.tensor(chunk[1:].reshape(1, -1), device=device)
        with torch.no_grad():
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                losses.append(model(x, y).item())
    return np.mean(losses)

val_v1 = eval_val(model_v1, val_data, SEQ_LEN)
val_v2 = eval_val(model_v2, val_data, SEQ_LEN)

bpb_v1 = val_v1 / math.log(2)
bpb_v2 = val_v2 / math.log(2)

print(f'\n  Itchy v1 (no tricks):')
print(f'    params:   {n_params(model_v1):,}')
print(f'    val_loss: {val_v1:.4f} nats')
print(f'    val_bpb:  {bpb_v1:.4f}')

print(f'\n  Itchy v2 (all tricks):')
print(f'    params:   {n_params(model_v2):,}')
print(f'    val_loss: {val_v2:.4f} nats')
print(f'    val_bpb:  {bpb_v2:.4f}')

print(f'\n  V2 vs V1 delta: {bpb_v2 - bpb_v1:+.4f} BPB (negative = v2 wins)')
print(f'  V2 vs V1 improvement: {(bpb_v1 - bpb_v2) / bpb_v1 * 100:.1f}%')